# JupyterLite（xeus-r）で学ぶ R 文法 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** の R カーネル（**xeus-r**）を使って、
統計解析言語 **R** の基本的な文法を一から学ぶためのチュートリアルです。

## 対象者
- プログラミングをこれから始める方
- R の文法を基礎から学び直したい方
- 統計テスト演習（`jupyterlite/jupyterlite_xeus_r_stats_practice.ipynb`）に進む前に、R そのものに慣れておきたい方

## このチュートリアルで学ぶこと
0. JupyterLite で R を使う準備（カーネルの選び方・パッケージの読み込み方・日本語とグラフの設定）
1. はじめての R（print と計算）
2. 変数とデータ型
3. ベクトル
4. 文字列
5. リスト・データフレーム・因子
6. 条件分岐（if 文）
7. 繰り返し（for 文・while 文・apply 系関数）
8. 関数
9. エラーと警告
10. 乱数・統計・日付
11. ファイルの読み書き
12. グラフの基本
13. まとめと総合演習

## 使い方
- 右上のカーネル名が **R**（`R 4.5.1 (xr)` など）になっていることを確認してください。「Select Kernel」のダイアログが出たら R を選びます。
- セルを上から順に `Shift + Enter` で実行してください。途中を飛ばすと、変数が未定義でエラーになることがあります。
- 各章の最後に **練習問題** があります。「解答欄」のセルに自分でコードを書いてから、「解答例」を開いて確認しましょう。

---
## 0. JupyterLite で R を使う準備

### 0.1 R カーネル（xeus-r）とは

JupyterLite では、R を WebAssembly 向けにビルドしたもの（**xeus-r**）が **ブラウザの中で** 動きます。
R のインストールは不要ですが、通常の R / RStudio とはいくつか違いがあります。

| 項目 | 通常の R / RStudio | JupyterLite（xeus-r） |
|---|---|---|
| R の実行場所 | PC 上の R | ブラウザ内の WebAssembly 版 R |
| 起動 | すぐ | 初回は R 本体をダウンロードするため、数十秒かかることがある |
| パッケージの追加 | `install.packages()` | **使えない**。サイトのビルド時に組み込まれたものだけ使える（0.2 節） |
| ファイルの保存先 | PC のディスク | ブラウザのローカルストレージ |
| グラフ | ウィンドウや Plots ペインに表示 | セルの下に画像として表示（0.3 節） |

#### R カーネルの選び方
- このノートブックを開くと、自動的に R カーネルが割り当てられます（画面右上に `R 4.5.1 (xr)` のように表示されます）
- 「Select Kernel」というダイアログが出たときは、一覧から **R** を選んでください
- 新しく R のノートブックを作るときは、Launcher（`+` ボタン）の **Notebook** から R のアイコンを選びます
- カーネルの状態は画面下部に表示されます（`Idle` = 待機中、`Busy` = 実行中）

### 0.2 パッケージの読み込み方

R では `library(パッケージ名)` でパッケージを読み込みます。JupyterLite の R カーネルで使えるのは、
**サイトをビルドするときに組み込まれたパッケージだけ** です。このサイトの R 環境には、
R 本体に付属する `base`、`stats`、`graphics`、`utils`、`datasets` などと、Jupyter 表示用のパッケージ
（`repr`、`IRdisplay`、`jsonlite` など）が含まれています。

```r
library(stats)      # 付属パッケージは読み込める（stats は最初から読み込み済み）
library(ggplot2)    # 組み込まれていないパッケージはエラーになる
# Error in library(ggplot2) : there is no package called ‘ggplot2’
```

#### `install.packages()` は使えません

ブラウザの中では CRAN からパッケージをダウンロードしてコンパイルすることができないため、
`install.packages("ggplot2")` のような追加インストールはできません。
パッケージを追加したい場合は、サイトの管理者が `environment.yml` に `r-ggplot2` のように書き加えて
サイトをビルドし直す必要があります（emscripten-forge で配布されているパッケージに限られます）。

#### 使えるパッケージを確認する

`installed.packages()` で、いま使えるパッケージの一覧が得られます。
`requireNamespace()` を使うと、エラーを出さずに「あるかどうか」だけを調べられます。

それでは確認してみましょう。あわせて、日付関数の警告を防ぐためにタイムゾーンも設定しておきます。

In [ ]:
# タイムゾーンの設定（JupyterLite では未設定のため、Sys.Date() などが警告を出すのを防ぐ）
Sys.setenv(TZ = "Asia/Tokyo")

cat("R のバージョン:", R.version.string, "\n")
cat("実行環境      :", Sys.info()[["sysname"]], "\n")   # JupyterLite では "Emscripten" と表示される

pkgs <- rownames(installed.packages())
cat("使えるパッケージの数:", length(pkgs), "\n")
print(pkgs)

In [ ]:
# パッケージがあるかどうかをエラーなしで調べる
cat("stats   :", requireNamespace("stats", quietly = TRUE), "\n")
cat("jsonlite:", requireNamespace("jsonlite", quietly = TRUE), "\n")
cat("ggplot2 :", requireNamespace("ggplot2", quietly = TRUE), "\n")   # このサイトでは FALSE

### 0.3 日本語の扱いとグラフの設定

#### 文字列の日本語
R カーネルは UTF-8 で動いているので、文字列の中の日本語はそのまま扱えます。

In [ ]:
print(Sys.getlocale("LC_CTYPE"))          # "en_US.UTF-8" のように UTF-8 であれば OK
print("こんにちは、R!")
cat("文字数:", nchar("こんにちは"), "\n")
paste("東京", "名古屋", "大阪", sep = " → ")

#### グラフの大きさと表示倍率

グラフはセルの下に PNG 画像として表示されます。大きさと解像度は `options()` で指定できます。

| オプション | 意味 | 既定値 |
|---|---|---|
| `repr.plot.width`, `repr.plot.height` | 画像の幅と高さ（インチ） | 7, 7 |
| `repr.plot.res` | 解像度（dpi） | 120 |
| `jupyter.plot_scale` | 表示するときの縮小率（2 なら半分の大きさで表示） | 2 |

既定では 7 インチ四方の画像が **半分の大きさに縮小して表示される** ため、文字がかなり小さく見えます。
このノートブックでは、横長・等倍表示に設定します。この設定は **カーネルを再起動すると元に戻る** ので、
グラフを描く前に毎回実行してください。

```r
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 100, jupyter.plot_scale = 1)
```

なお、現在の WebAssembly 版 R では、`cex.main` や `par(ps = ...)` などの **文字サイズの指定が反映されず、
文字は小さめの固定サイズで描かれます**（点や線の大きさは変えられます）。文字の大きさを変えたい場合は、
画像そのものを大きくしてください（`repr.plot.width` / `repr.plot.height` を大きくする）。

#### グラフの中の日本語について（重要）

現在の JupyterLite の R 環境には **日本語フォントが含まれていません**。そのため、`main = "売上の推移"` のように
グラフのタイトルや軸ラベルに日本語を使うと、文字が **□（豆腐）** になります。

このノートブックでは、次の方針でグラフを描きます。

- グラフの中のタイトル・軸ラベル・凡例は **英語（またはローマ字）** で書く
- 日本語の説明は Markdown セルや `cat()` の出力で補う

Python 側の環境では `japanize-matplotlib-jlite` で日本語グラフが描けるので、日本語入りのグラフが必要な場合は
Python のチュートリアル（`python/python_beginner_tutorial.ipynb`）も参照してください。

In [ ]:
# グラフの大きさを設定して、簡単なグラフを描いてみる（ラベルは英語で）
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 100, jupyter.plot_scale = 1)

months <- 1:6
sales <- c(120, 135, 150, 128, 170, 190)
plot(months, sales, type = "b", main = "Monthly sales", xlab = "Month", ylab = "Sales (10k yen)")

### 0.4 この先の章について

第 1 章から第 11 章までは R の基本文法だけを扱います。第 12 章と総合演習で、上の設定を使ってグラフを描きます。

---
## 1. はじめての R

### 1.1 print と cat で表示する

- `print()`：値を R の形式で表示する（文字列は `"` 付き、先頭に `[1]` が付く）
- `cat()`：値をそのまま並べて表示する（改行は `\n` で自分で入れる）

`[1]` は「この行の最初の要素が 1 番目」という意味で、R がベクトル（第 3 章）を表示するときの目印です。

**Jupyter での表示ルール（重要）**：変数名や式をそのまま書くと値が表示されますが、JupyterLite の R カーネルでは
**セルの最後の式の値だけ** が自動表示されます。途中の結果も見たいときは `print()` や `cat()` で明示的に表示してください。
このノートブックでも、1 つのセルで複数の結果を見せるときは `print()` を使っています。

In [ ]:
print("こんにちは、R!")
cat("こんにちは、R!\n")
cat("R", "は", "楽しい", "\n")             # スペースで区切って表示
cat("2026", "08", "30", sep = "-")         # sep で区切り文字を指定
cat("\n")

### 1.2 コメント

`#` から行末までは **コメント** として無視されます。

In [ ]:
# これはコメントです。実行されません
print("コメントは無視されます")   # 行の途中からでも書けます

### 1.3 R を電卓として使う

| 演算子 | 意味 | 例 | 結果 |
|:---:|---|---|---|
| `+` | 足し算 | `7 + 3` | `10` |
| `-` | 引き算 | `7 - 3` | `4` |
| `*` | 掛け算 | `7 * 3` | `21` |
| `/` | 割り算 | `7 / 2` | `3.5` |
| `%/%` | 割り算の商（切り捨て） | `7 %/% 2` | `3` |
| `%%` | 割り算の余り | `7 %% 2` | `1` |
| `^` | べき乗 | `2 ^ 10` | `1024` |

In [ ]:
print(7 + 3)
print(7 - 3)
print(7 * 3)
print(7 / 2)
print(7 %/% 2)     # 商
print(7 %% 2)      # 余り
2 ^ 10      # べき乗

In [ ]:
# 計算の優先順位は数学と同じ。かっこで変更できる
print(2 + 3 * 4)
print((2 + 3) * 4)
print(sqrt(16))          # 平方根
print(abs(-5))           # 絶対値
round(3.14159, 2) # 四捨五入（小数第 2 位まで）

上のセルでは、途中の結果は `print()` で表示し、最後の `round(3.14159, 2)` だけは式をそのまま書いて自動表示させています。
代入（第 2 章）だけの行は何も表示しません。

### 練習問題 1

1. `cat()` を使って「太郎さん、こんにちは」のように、自分の名前を含む挨拶を表示してください。
2. 1 週間は何秒か計算してください（1 日 = 24 時間、1 時間 = 60 分、1 分 = 60 秒）。
3. 1234 を 7 で割ったときの商と余りを求めてください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```r
# 1
cat("太郎さん、こんにちは\n")

# 2
print(7 * 24 * 60 * 60)

# 3
print(1234 %/% 7)
1234 %% 7
```

</details>

---
## 2. 変数とデータ型

### 2.1 変数と代入

R では `<-` で変数に値を代入します（`=` も使えますが、R では `<-` が一般的です）。

In [ ]:
price <- 120          # 変数 price に 120 を代入
quantity <- 3
total <- price * quantity
total                  # 変数名だけ書くと値が表示される

In [ ]:
count <- 1
count <- count + 1     # 今の値に 1 を足して入れ直す（R には += はありません）
count

#### 変数名のルール
- 英字・数字・ピリオド `.`・アンダースコア `_` が使える（先頭は英字が基本）
- 大文字と小文字は区別される（`Total` と `total` は別の変数）
- `if`、`for`、`function`、`TRUE` などの予約語は使えない
- `total_price` や `total.price` のように、意味のわかる名前を付ける

### 2.2 データ型

`class()` で値の型（クラス）を確認できます。

| 型 | 説明 | 例 |
|---|---|---|
| `numeric` | 数値（小数も整数も基本はこれ） | `20`, `170.5` |
| `integer` | 整数（末尾に `L` を付ける） | `20L` |
| `character` | 文字列 | `"山田"` |
| `logical` | 論理値（真偽値） | `TRUE`, `FALSE` |

In [ ]:
age <- 20
height <- 170.5
name <- "山田"
is_student <- TRUE

print(class(age))
print(class(height))
print(class(name))
print(class(is_student))
class(20L)            # 整数

In [ ]:
# 型を調べる関数（TRUE / FALSE を返す）
print(is.numeric(age))
print(is.character(name))
is.logical(is_student)

### 2.3 型変換

`as.numeric()`、`as.character()`、`as.integer()`、`as.logical()` で型を変換できます。
数値に変換できない文字列を `as.numeric()` に渡すと、**警告** が出て `NA`（欠損値）になります。

In [ ]:
print(as.numeric("100") + 50)        # 文字列 → 数値
print(as.character(42))              # 数値 → 文字列
print(as.integer(3.99))              # 小数 → 整数（切り捨て）
print(as.logical("TRUE"))            # 文字列 → 論理値
as.numeric(TRUE)              # TRUE は 1、FALSE は 0

### 2.4 NA と NULL

- `NA`：**欠損値**（値があるはずなのに分からない）。計算に混ざると結果も `NA` になる
- `NULL`：**何もない** ことを表す特別な値

In [ ]:
x <- c(10, 20, NA, 40)
print(mean(x))                       # NA が混ざると NA になる
print(mean(x, na.rm = TRUE))         # na.rm = TRUE で NA を除いて計算
print(is.na(x))                      # どれが NA かを調べる

# 変換できない文字列は NA になる（警告を抑えるために suppressWarnings で囲んでいます）
suppressWarnings(as.numeric("abc"))

### 練習問題 2

1. 半径 5 の円の面積を、変数 `radius` と `area` を使って計算してください（円周率は `pi` という組み込みの定数が使えます）。
2. 文字列 `"25"` と `"17"` をそれぞれ数値に変換してから足し算してください。
3. `c(5, NA, 15)` の平均を、NA を除いて計算してください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```r
# 1
radius <- 5
area <- pi * radius ^ 2
print(area)

# 2
print(as.numeric("25") + as.numeric("17"))

# 3
mean(c(5, NA, 15), na.rm = TRUE)
```

</details>

---
## 3. ベクトル

**ベクトル** は同じ型の値を並べたもので、R の最も基本的なデータ構造です。
実は、これまで使ってきた `20` や `"山田"` も「長さ 1 のベクトル」です。

### 3.1 ベクトルの作成

In [ ]:
scores <- c(72, 95, 58, 88, 64)      # c() で値をまとめる
fruits <- c("りんご", "バナナ", "みかん")
print(scores)
print(fruits)
length(scores)                        # 要素の数

In [ ]:
print(1:10)                       # 1 から 10 までの整数
print(seq(0, 100, by = 25))       # 0 から 100 まで 25 刻み
print(seq(1, 2, length.out = 5))  # 1 から 2 までを 5 等分
print(rep("A", 3))                # 繰り返し
rep(c(1, 2), times = 3)

### 3.2 インデックス（要素の取り出し）

R のインデックスは **1 から始まります**（Python などの 0 始まりとは違うので注意）。
負の数を指定すると「その要素を除く」という意味になります。

In [ ]:
scores <- c(72, 95, 58, 88, 64)
print(scores[1])          # 1 番目
print(scores[5])          # 5 番目（最後）
print(scores[2:4])        # 2〜4 番目
print(scores[c(1, 3)])    # 1 番目と 3 番目
scores[-1]         # 1 番目を除く

In [ ]:
# 条件に合う要素だけを取り出す（論理ベクトルによるインデックス）
print(scores > 70)                # 各要素が 70 より大きいか（TRUE / FALSE のベクトル）
print(scores[scores > 70])        # TRUE の位置の要素だけ
which(scores > 70)         # TRUE の位置（番号）

### 3.3 ベクトル演算

ベクトルに対する計算は、**要素ごとに一括で** 行われます（ループを書く必要がありません）。

In [ ]:
print(scores * 2)                  # 全要素を 2 倍
print(scores + c(10, 10, 10, 10, 10))
print(scores + 10)                 # 長さ 1 のベクトルは自動的に繰り返される（リサイクル）
sqrt(c(4, 9, 16))

### 3.4 便利な関数

In [ ]:
print(sum(scores))           # 合計
print(mean(scores))          # 平均
print(max(scores))           # 最大値
print(min(scores))           # 最小値
print(sort(scores))          # 昇順に並べ替え
print(sort(scores, decreasing = TRUE))   # 降順
print(rev(scores))           # 逆順
print(order(scores))         # 並べ替えたときの元の位置
cumsum(scores)        # 累積和

In [ ]:
# 要素に名前を付ける
names(scores) <- c("田中", "鈴木", "佐藤", "高橋", "伊藤")
print(scores)
print(scores["鈴木"])

# 含まれているかどうか
print("バナナ" %in% fruits)
c(1, 2, 3) %in% c(2, 3, 4)

### 練習問題 3

1. 1 から 10 までの整数のベクトルを作り、偶数だけを取り出してください。
2. `c(3, 1, 4, 1, 5, 9, 2, 6)` の最大値・最小値・合計・平均を求めてください。
3. 同じベクトルを降順に並べ替えて、上位 3 つだけを表示してください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```r
# 1
v <- 1:10
print(v[v %% 2 == 0])

# 2
x <- c(3, 1, 4, 1, 5, 9, 2, 6)
print(max(x))
print(min(x))
print(sum(x))
print(mean(x))

# 3
sort(x, decreasing = TRUE)[1:3]
```

</details>

---
## 4. 文字列

### 4.1 連結と長さ

- `paste()`：文字列を連結する（区切りは `sep`、既定はスペース）
- `paste0()`：区切りなしで連結する
- `nchar()`：文字数

In [ ]:
first <- "R"
second <- "入門"
print(paste(first, second))
print(paste0(first, second))
print(paste("東京", "名古屋", "大阪", sep = ", "))
print(nchar("Python"))
nchar("日本語も1文字ずつ数える")

In [ ]:
# paste はベクトルにも使える（要素ごとに連結）
names <- c("田中", "鈴木", "佐藤")
print(paste(names, "さん"))
paste0(names, "さん", collapse = "・")   # collapse で 1 つの文字列にまとめる

### 4.2 部分文字列と大文字・小文字

In [ ]:
word <- "Python"
print(substr(word, 1, 3))       # 1 文字目から 3 文字目まで
print(substr(word, 4, 6))
print(toupper(word))            # 大文字に
tolower(word)            # 小文字に

### 4.3 分割・置換・検索

- `strsplit()`：分割する（結果はリストなので `[[1]]` で取り出す）
- `sub()` / `gsub()`：最初の 1 つ / すべてを置き換える
- `grepl()`：含まれているかどうか（TRUE / FALSE）
- `trimws()`：前後の空白を取り除く

In [ ]:
parts <- strsplit("2026-08-30", "-")[[1]]
print(parts)
print(parts[1])

text <- "  Hello, R World  "
print(trimws(text))
print(gsub("o", "0", text))                 # すべての o を 0 に
print(sub("o", "0", text))                  # 最初の o だけ
print(grepl("World", text))                 # 含まれているか
print(startsWith("data.csv", "data"))
endsWith("data.csv", ".csv")

### 4.4 書式付きの文字列（sprintf、format）

`sprintf()` は C 言語由来の書式指定です。`%s` は文字列、`%d` は整数、`%.2f` は小数点以下 2 桁の数値に置き換わります。

In [ ]:
name <- "佐藤"
age <- 21
print(sprintf("%sさんは%d歳です", name, age))
print(sprintf("円周率: %.2f", pi))
print(sprintf("%5.1f%%", 25.678))             # 幅 5、小数 1 桁、% 記号は %% で書く
print(format(1234567, big.mark = ","))        # 3 桁区切り
format(Sys.Date(), "%Y年%m月%d日")     # 日付の書式（第 10 章）

### 練習問題 4

1. `s <- "Hello, World"` から `substr()` で `"World"` を取り出してください。
2. `"2026-08-30"` を `"-"` で分割して、年・月・日をそれぞれ数値として取り出してください。
3. 商品名 `"りんご"`、単価 `128`、個数 `3` を変数にして、`sprintf()` で `りんご 3個: 384円` と表示してください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```r
# 1
s <- "Hello, World"
print(substr(s, 8, 12))

# 2
parts <- as.numeric(strsplit("2026-08-30", "-")[[1]])
print(parts[1])
print(parts[2])
print(parts[3])

# 3
item <- "りんご"
price <- 128
count <- 3
sprintf("%s %d個: %d円", item, count, price * count)
```

</details>

---
## 5. リスト・データフレーム・因子

| 種類 | 説明 |
|---|---|
| ベクトル | 同じ型の値の並び |
| リスト `list` | 異なる型の値をまとめられる。名前を付けられる |
| データフレーム `data.frame` | 表形式のデータ（各列がベクトル）。データ分析の中心 |
| 因子 `factor` | カテゴリ（分類）を表す型 |
| 行列 `matrix` | 同じ型の値を縦横に並べたもの |

### 5.1 リスト

In [ ]:
student <- list(name = "田中", age = 20, scores = c(80, 75, 90))
print(student$name)              # $ で要素を取り出す
print(student[["scores"]])       # [[ ]] でも取り出せる
student$age <- 21         # 変更
student$major <- "経済学"  # 追加
print(names(student))
print(length(student))
str(student)              # 構造をまとめて表示

### 5.2 データフレーム

データフレームは「列ごとに型の異なる表」です。CSV ファイルを読み込むとデータフレームになります（第 11 章）。

In [ ]:
df <- data.frame(
  name = c("田中", "鈴木", "佐藤", "高橋"),
  kokugo = c(78, 88, 95, 55),
  sugaku = c(92, 64, 89, 71),
  eigo = c(85, 71, 93, 60)
)
df

In [ ]:
print(nrow(df))          # 行数
print(ncol(df))          # 列数
print(dim(df))           # 行数と列数
print(colnames(df))      # 列名
str(df)           # 構造
summary(df)       # 要約統計量

In [ ]:
# 列の取り出し
print(df$kokugo)
print(df[["sugaku"]])
print(mean(df$eigo))

# 行と列の指定：df[行, 列]
print(df[1, ])                  # 1 行目
print(df[, "name"])             # name 列
print(df[2, "sugaku"])          # 2 行目の sugaku 列
df[df$kokugo >= 80, ]    # 条件に合う行だけ

In [ ]:
# 列の追加と並べ替え
df$total <- df$kokugo + df$sugaku + df$eigo
df$avg <- round(df$total / 3, 1)
print(df)
df[order(df$total, decreasing = TRUE), ]   # 合計の降順に並べ替え

In [ ]:
# R に付属するサンプルデータ（iris：アヤメの花のデータ）
print(head(iris))             # 先頭 6 行
table(iris$Species)    # 種類ごとの件数

### 5.3 因子（factor）

カテゴリ（分類）を表すデータは `factor()` にすると、`table()` などで集計しやすくなり、
`levels` で順序を指定することもできます。

In [ ]:
size <- c("中", "小", "大", "中", "小", "中")
size_f <- factor(size, levels = c("小", "中", "大"))
print(size_f)
print(levels(size_f))
table(size_f)

### 5.4 行列（matrix）

In [ ]:
m <- matrix(1:6, nrow = 2)     # 2 行 3 列（列方向に順に埋まる）
print(m)
print(m[2, 3])                        # 2 行目の 3 列目
print(t(m))                           # 転置
dim(m)

### 練習問題 5

1. 商品名（`"りんご"`, `"バナナ"`, `"みかん"`）と単価（128, 98, 60）と個数（3, 5, 10）の列を持つデータフレーム `items` を作ってください。
2. `items` に「金額 = 単価 × 個数」の列 `amount` を追加し、金額の合計を求めてください。
3. 金額が 300 以上の行だけを表示してください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```r
# 1
items <- data.frame(
  name = c("りんご", "バナナ", "みかん"),
  price = c(128, 98, 60),
  count = c(3, 5, 10)
)
print(items)

# 2
items$amount <- items$price * items$count
print(sum(items$amount))

# 3
items[items$amount >= 300, ]
```

</details>

---
## 6. 条件分岐（if 文）

### 6.1 比較演算子と論理値

| 演算子 | 意味 |
|:---:|---|
| `==` | 等しい（`=` 1 つは代入なので注意） |
| `!=` | 等しくない |
| `<` `<=` | より小さい、以下 |
| `>` `>=` | より大きい、以上 |
| `&` `|` `!` | かつ、または、否定（ベクトルの要素ごと） |
| `&&` `||` | かつ、または（長さ 1 の条件用。`if` の条件ではこちらを使う） |

In [ ]:
x <- 10
print(x > 5)
print(x == 10)
print(x != 10)
print(x >= 5 && x <= 15)       # かつ
print(x < 0 || x > 100)        # または
!(x > 5)                # 否定

### 6.2 if / else if / else

```r
if (条件1) {
  条件1 が TRUE のときの処理
} else if (条件2) {
  条件2 が TRUE のときの処理
} else {
  どれにも当てはまらないときの処理
}
```

- 条件は `( )` で囲み、処理は `{ }` で囲みます
- `else` は **`}` と同じ行に** 書きます（行を分けるとエラーになります）

In [ ]:
score <- 78
if (score >= 80) {
  print("優")
} else if (score >= 60) {
  print("良")
} else {
  print("不可")
}

In [ ]:
temperature <- 28
if (temperature >= 25) {
  cat("暑い日です\n")
  cat("水分補給を忘れずに\n")     # 同じ { } の中なので、同じブロック
}
cat("今日の気温は", temperature, "度\n")    # if の外なので、常に実行される

### 6.3 ifelse()：ベクトルに対する条件分岐

`if` は条件が 1 つの値のときに使います。ベクトルの要素ごとに値を選びたいときは `ifelse(条件, TRUE のときの値, FALSE のときの値)` を使います。

In [ ]:
scores <- c(72, 95, 58, 88, 64)
print(ifelse(scores >= 70, "合格", "不合格"))

n <- 7
parity <- ifelse(n %% 2 == 0, "偶数", "奇数")
cat(n, "は", parity, "\n")

### 6.4 switch()：値に応じて分岐

In [ ]:
day <- "sat"
kind <- switch(day,
  sat = "週末",
  sun = "週末",
  "平日"              # どれにも当てはまらないとき
)
kind

### 練習問題 6

1. 変数 `year` がうるう年かどうかを判定して表示してください（4 で割り切れ、かつ 100 で割り切れない年、または 400 で割り切れる年がうるう年です）。
2. 身長 `height`（m）と体重 `weight`（kg）から BMI（体重 ÷ 身長²）を計算し、18.5 未満なら「低体重」、25 未満なら「普通体重」、それ以上なら「肥満」と表示してください。
3. `c(55, 70, 82, 91, 45)` の各要素について、60 以上なら "pass"、そうでなければ "fail" のベクトルを作ってください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```r
# 1
year <- 2024
if ((year %% 4 == 0 && year %% 100 != 0) || year %% 400 == 0) {
  cat(year, "年はうるう年です\n")
} else {
  cat(year, "年はうるう年ではありません\n")
}

# 2
height <- 1.70
weight <- 65
bmi <- weight / height ^ 2
if (bmi < 18.5) {
  print("低体重")
} else if (bmi < 25) {
  print("普通体重")
} else {
  print("肥満")
}

# 3
x <- c(55, 70, 82, 91, 45)
ifelse(x >= 60, "pass", "fail")
```

</details>

---
## 7. 繰り返し（for 文・while 文・apply 系関数）

### 7.1 for 文

`for (変数 in ベクトル) { 処理 }` と書くと、ベクトルの要素を 1 つずつ取り出して処理を繰り返します。

In [ ]:
for (i in 1:5) {
  print(i)
}

In [ ]:
total <- 0
for (i in 1:10) {
  total <- total + i
}
cat("1 から 10 の合計:", total, "\n")

In [ ]:
fruits <- c("りんご", "バナナ", "みかん")
for (fruit in fruits) {
  cat(fruit, "は", nchar(fruit), "文字\n")
}

# 番号も使いたいときは seq_along()
for (i in seq_along(fruits)) {
  cat(i, "番目:", fruits[i], "\n")
}

### 7.2 while 文

`while (条件) { 処理 }` は、条件が `TRUE` の間ずっと繰り返します。
条件がいつまでも `FALSE` にならないと **無限ループ** になるので注意してください
（止まらなくなったら、メニューの Kernel → Interrupt Kernel で中断できます）。

In [ ]:
count <- 0
while (count < 3) {
  cat("count =", count, "\n")
  count <- count + 1
}
cat("終了\n")

In [ ]:
# 例：年利 5% で預金が 2 倍になるまでの年数
balance <- 100
years <- 0
while (balance < 200) {
  balance <- balance * 1.05
  years <- years + 1
}
cat(years, "年後に", round(balance, 1), "になりました\n")

### 7.3 break と next

- `break`：ループを途中で終了する
- `next`：今回の残りの処理を飛ばして次の繰り返しへ進む（Python の `continue` に相当）

In [ ]:
for (i in 1:10) {
  if (i == 7) {
    break            # ループを抜ける
  }
  if (i %% 2 == 0) {
    next             # 偶数のときは以降をスキップ
  }
  print(i)
}

### 7.4 ネストしたループ

In [ ]:
for (i in 1:3) {
  for (j in 1:3) {
    cat(sprintf("%3d", i * j))
  }
  cat("\n")     # 1 行分が終わったら改行
}

### 7.5 ループを書かずに済ませる：ベクトル演算と apply 系関数

R では、ループより **ベクトル演算** や **`sapply()` / `lapply()`** を使う方が簡潔で速いことが多いです。

- `sapply(ベクトル, 関数)`：各要素に関数を適用し、結果をベクトルで返す
- `lapply(ベクトル, 関数)`：同じだが、結果をリストで返す
- `apply(行列やデータフレーム, 1 または 2, 関数)`：行（1）または列（2）ごとに関数を適用する

In [ ]:
# 1〜5 の 2 乗
squares <- c()
for (i in 1:5) {
  squares <- c(squares, i ^ 2)
}
print(squares)

print((1:5) ^ 2)                     # ベクトル演算なら 1 行
sapply(1:5, function(i) i ^ 2)   # sapply でも同じ

In [ ]:
df <- data.frame(kokugo = c(78, 88, 95), sugaku = c(92, 64, 89), eigo = c(85, 71, 93))
print(apply(df, 1, sum))     # 行ごとの合計（各学生の合計点）
print(apply(df, 2, mean))    # 列ごとの平均（各科目の平均点）
print(colMeans(df))          # 列ごとの平均は専用の関数もある
rowSums(df)           # 行ごとの合計

### 練習問題 7

1. `for` 文を使って 1 から 100 までの偶数の合計を求めてください。
2. 1 から 15 までの数について、3 の倍数なら「Fizz」、5 の倍数なら「Buzz」、両方の倍数なら「FizzBuzz」、それ以外はその数を表示してください（FizzBuzz 問題）。
3. 九九の 5 の段を `5 x 1 = 5` の形式で表示してください。
4. `sapply()` を使って、1 から 10 までの各数の 3 乗のベクトルを作ってください。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```r
# 1
total <- 0
for (i in 1:100) {
  if (i %% 2 == 0) {
    total <- total + i
  }
}
print(total)

# 2
for (i in 1:15) {
  if (i %% 15 == 0) {
    print("FizzBuzz")
  } else if (i %% 3 == 0) {
    print("Fizz")
  } else if (i %% 5 == 0) {
    print("Buzz")
  } else {
    print(i)
  }
}

# 3
for (i in 1:9) {
  cat(sprintf("5 x %d = %d\n", i, 5 * i))
}

# 4
sapply(1:10, function(x) x ^ 3)
```

</details>

---
## 8. 関数

**関数** は、ひとまとまりの処理に名前を付けて再利用できるようにしたものです。

```r
関数名 <- function(引数1, 引数2, ...) {
  処理
  return(戻り値)
}
```

`return()` を省略すると、**最後に評価した式の値** が戻り値になります。

### 8.1 関数の定義と呼び出し

In [ ]:
greet <- function() {
  cat("こんにちは！\n")
}

print(greet())
greet()

In [ ]:
greet <- function(name) {              # name は引数
  cat("こんにちは、", name, "さん！\n", sep = "")
}

print(greet("山田"))
greet("鈴木")

### 8.2 戻り値

In [ ]:
add <- function(a, b) {
  return(a + b)
}

result <- add(3, 4)
print(result)
add(10, 20) * 2

In [ ]:
circle_area <- function(radius) {
  pi * radius ^ 2      # return() を省略すると、最後の式の値が返る
}

print(circle_area(2))
circle_area(c(1, 2, 3))     # ベクトルを渡せば、要素ごとに計算される

### 8.3 デフォルト引数と名前付き引数

In [ ]:
introduce <- function(name, age = 20, city = "名古屋") {
  cat(sprintf("%s（%d歳、%s在住）\n", name, age, city))
}

print(introduce("田中"))
print(introduce("鈴木", 25))
print(introduce("佐藤", city = "東京"))
introduce(age = 30, name = "高橋")

### 8.4 複数の値を返す

R の関数は 1 つの値しか返せませんが、リストやベクトルにまとめれば複数の値を返せます。

In [ ]:
min_max <- function(values) {
  list(min = min(values), max = max(values))
}

result <- min_max(c(3, 8, 1, 9, 4))
print(result$min)
result$max

### 8.5 変数のスコープ（有効範囲）

関数の中で作った変数は関数の外からは見えません。関数の外の変数は、関数の中から読むことはできます。

In [ ]:
message_text <- "グローバル変数"

show <- function() {
  message_text <- "ローカル変数"     # 関数の中だけで有効な別の変数
  cat("関数の中:", message_text, "\n")
}

print(show())
cat("関数の外:", message_text, "\n")   # 外の変数は変わっていない

### 8.6 無名関数

`sapply()` などに渡す小さな関数は、名前を付けずにその場で書けます。
R 4.1 以降では `\(x) x ^ 2` という短い書き方もできます。

In [ ]:
print(sapply(1:5, function(x) x ^ 2))
print(sapply(1:5, \(x) x ^ 2))          # 短い書き方（R 4.1 以降）

# sort と order で並べ替えの基準を指定する例
people <- data.frame(name = c("田中", "鈴木", "佐藤"), age = c(25, 19, 32))
people[order(people$age), ]

### 8.7 よく使う組み込み関数

| 関数 | 説明 |
|---|---|
| `print()`, `cat()`, `paste()`, `sprintf()` | 表示・文字列 |
| `length()`, `class()`, `str()`, `summary()` | 長さ、型、構造、要約 |
| `as.numeric()`, `as.character()`, `as.integer()` | 型変換 |
| `sum()`, `mean()`, `max()`, `min()`, `sort()`, `rev()` | 集計・並べ替え |
| `abs()`, `round()`, `sqrt()`, `exp()`, `log()` | 数学関数 |
| `seq()`, `rep()`, `seq_along()` | ベクトル生成 |
| `sapply()`, `lapply()`, `apply()` | 関数の一括適用 |
| `help(関数名)` または `?関数名` | ヘルプの表示 |

### 練習問題 8

1. 摂氏温度を華氏温度に変換する関数 `c_to_f(celsius)` を作り、`c_to_f(25)` の結果を表示してください（華氏 = 摂氏 × 9 / 5 + 32）。
2. ベクトルを受け取って平均を返す関数 `average(values)` を作ってください。長さが 0 のときは `0` を返します。
3. 税込価格を返す関数 `with_tax(price, rate = 0.1)` を作り、`with_tax(1000)` と `with_tax(1000, rate = 0.08)` を表示してください。

In [ ]:
# 練習問題 8 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 8 の解答例を見る</strong></summary>

```r
# 1
c_to_f <- function(celsius) {
  celsius * 9 / 5 + 32
}
print(c_to_f(25))

# 2
average <- function(values) {
  if (length(values) == 0) {
    return(0)
  }
  sum(values) / length(values)
}
print(average(c(80, 90, 70)))
print(average(c()))

# 3
with_tax <- function(price, rate = 0.1) {
  floor(price * (1 + rate))
}
print(with_tax(1000))
with_tax(1000, rate = 0.08)
```

</details>

---
## 9. エラーと警告

### 9.1 エラーメッセージの読み方

R のエラーは `Error in 〜 : 説明` の形で表示されます。`in` の後ろがエラーの起きた場所、`:` の後ろが原因です。

| メッセージ | 主な原因 |
|---|---|
| `object 'x' not found` | 定義していない変数を使った（実行順やスペルミスに注意） |
| `could not find function "f"` | 存在しない関数名。パッケージの読み込み忘れやスペルミス |
| `unexpected symbol` / `unexpected '}'` | 文法の間違い（かっこの閉じ忘れ、カンマ忘れなど） |
| `non-numeric argument to binary operator` | 数値でないもの（文字列など）を計算しようとした |
| `subscript out of bounds` | リストや行列の範囲外を指定した |
| `argument "x" is missing, with no default` | 必要な引数を渡していない |
| `there is no package called 'xxx'` | パッケージが組み込まれていない（0.2 節を参照） |

**警告（Warning）** はエラーと違って処理は続行されますが、意図しない結果（`NA` が混ざるなど）のサインなので、内容を確認しましょう。

次のセルのコメントを外して実行すると、実際にエラーを見ることができます（確認したら元に戻してください）。

In [ ]:
# わざとエラーを起こしてみる（1 行ずつコメントを外して試してみましょう）
# print(undefined_variable)     # object 'undefined_variable' not found
# 1 + "2"                       # non-numeric argument to binary operator
# c(1, 2, 3)[[5]]               # subscript out of bounds
cat("エラーが出なければ、このメッセージが表示されます\n")

### 9.2 tryCatch()：エラーが起きても処理を続ける

`tryCatch(処理, error = function(e) 対処, warning = function(w) 対処, finally = 後始末)` の形で、
エラーや警告を捕まえて対処できます。`conditionMessage(e)` でメッセージを取り出せます。

In [ ]:
result <- tryCatch(
  {
    log(-1)                       # 警告が出る（NaN が生成される）
  },
  warning = function(w) {
    cat("警告を捕まえました:", conditionMessage(w), "\n")
    NA
  }
)
result

In [ ]:
to_number <- function(text) {
  tryCatch(
    {
      value <- as.numeric(text)
      if (is.na(value)) stop("数値に変換できません")
      value
    },
    warning = function(w) {
      cat(sprintf("'%s' は数値に変換できません（警告）\n", text))
      NA
    },
    error = function(e) {
      cat("エラー:", conditionMessage(e), "\n")
      NA
    },
    finally = {
      cat("--- 処理終了 ---\n")
    }
  )
}

print(to_number("42"))
to_number("abc")

### 9.3 stop() と warning()：自分でエラーや警告を出す

In [ ]:
set_age <- function(age) {
  if (age < 0) {
    stop("年齢は 0 以上で指定してください")
  }
  age
}

result <- tryCatch(set_age(-1), error = function(e) conditionMessage(e))
result

### 練習問題 9

1. 文字列のベクトル `c("10", "abc", "30", "4.5")` の各要素を数値に変換して合計してください。変換できない要素は表示してスキップします（`suppressWarnings()` と `is.na()` を使う方法でも、`tryCatch()` を使う方法でも構いません）。
2. リスト `d` とキー `key` を受け取り、キーがあればその値を、なければ `"不明"` を返す関数 `lookup(d, key)` を作ってください（`is.null()` が使えます）。

In [ ]:
# 練習問題 9 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 9 の解答例を見る</strong></summary>

```r
# 1
texts <- c("10", "abc", "30", "4.5")
total <- 0
for (t in texts) {
  value <- suppressWarnings(as.numeric(t))
  if (is.na(value)) {
    cat(t, "はスキップしました\n")
  } else {
    total <- total + value
  }
}
cat("合計:", total, "\n")

# 2
lookup <- function(d, key) {
  value <- d[[key]]
  if (is.null(value)) {
    return("不明")
  }
  value
}
capitals <- list("愛知県" = "名古屋市")
print(lookup(capitals, "愛知県"))
lookup(capitals, "岐阜県")
```

</details>

---
## 10. 乱数・統計・日付

R は統計解析のための言語なので、統計関数が最初から豊富に用意されています（`stats` パッケージ）。

### 10.1 乱数

`set.seed()` で種を固定すると、毎回同じ乱数の並びになり、結果を再現できます。

In [ ]:
set.seed(0)                          # 乱数の種を固定（再現性のため）
print(sample(1:6, 1))                       # 1〜6 から 1 つ（サイコロ）
print(sample(1:6, 10, replace = TRUE))      # 重複ありで 10 回
print(runif(3))                             # 0〜1 の一様乱数
print(rnorm(5, mean = 50, sd = 10))         # 平均 50、標準偏差 10 の正規乱数
print(sample(c("グー", "チョキ", "パー"), 1))
sort(sample(1:43, 6))                # 重複なしで 6 個（ロト 6）

### 10.2 基本的な統計量

In [ ]:
set.seed(42)
x <- rnorm(100, mean = 60, sd = 10)   # 100 人分のテスト得点（想定）

print(mean(x))             # 平均
print(median(x))           # 中央値
print(sd(x))               # 標準偏差
print(var(x))              # 分散
print(quantile(x))         # 四分位数
summary(x)          # まとめて表示

In [ ]:
# 2 つの変数の相関
study_hours <- c(1, 2, 3, 4, 5, 6, 7, 8)
scores <- c(52, 58, 61, 66, 70, 78, 80, 88)
print(cor(study_hours, scores))          # 相関係数
table(c("A", "B", "A", "C", "A"))  # 度数分布

### 10.3 日付と時刻

`Sys.Date()` は今日の日付、`Sys.time()` は現在時刻を返します。`as.Date()` で文字列を日付に変換すると、日数の計算ができます。

In [ ]:
today <- Sys.Date()
print(today)
print(format(today, "%Y年%m月%d日"))       # 書式を指定して文字列に
print(today + 100)                          # 100 日後
print(as.Date("2026-04-01") + 30)
print(as.numeric(as.Date("2026-12-31") - as.Date("2026-04-01")))   # 日数の差
print(weekdays(as.Date("2026-04-01")))      # 曜日（英語）
format(Sys.time(), "%H:%M:%S")       # 現在時刻

### 練習問題 10

1. `sqrt(2)` を小数第 3 位まで表示してください（`round()` または `sprintf()`）。
2. `set.seed(1)` のあと、サイコロを 10 回振った結果をベクトルに入れ、その平均を求めてください。
3. 今日から 30 日後の日付を `2026年09月29日` の形式で表示してください。

In [ ]:
# 練習問題 10 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 10 の解答例を見る</strong></summary>

```r
# 1
print(round(sqrt(2), 3))
print(sprintf("%.3f", sqrt(2)))

# 2
set.seed(1)
rolls <- sample(1:6, 10, replace = TRUE)
print(rolls)
print(mean(rolls))

# 3
format(Sys.Date() + 30, "%Y年%m月%d日")
```

</details>

---
## 11. ファイルの読み書き

JupyterLite で作成したファイルはブラウザのローカルストレージに保存され、左側のファイルブラウザに表示されます
（表示されないときは、ファイルブラウザ上部の更新ボタンを押してください）。

### 11.1 テキストファイル

- `writeLines(文字列ベクトル, ファイル名)`：1 要素を 1 行として書き込む
- `readLines(ファイル名)`：1 行を 1 要素とするベクトルとして読み込む

In [ ]:
lines <- c("1行目: R の練習", "2行目: ファイルに書き込み")
writeLines(lines, "sample_r.txt")
cat("sample_r.txt に書き込みました\n")

In [ ]:
content <- readLines("sample_r.txt")
print(content)
print(length(content))               # 行数

for (i in seq_along(content)) {
  cat(i, content[i], "\n")
}

In [ ]:
# 追記：既存の内容を読み込んで、行を足して書き直す
content <- c(readLines("sample_r.txt"), "3行目: 追記しました")
writeLines(content, "sample_r.txt")
cat(readLines("sample_r.txt"), sep = "\n")

### 11.2 CSV ファイル

表形式のデータは CSV（カンマ区切り）ファイルで扱うことが多く、`write.csv()` / `read.csv()` でデータフレームをそのまま読み書きできます。

In [ ]:
df <- data.frame(
  name = c("田中", "鈴木", "佐藤"),
  kokugo = c(80, 92, 67),
  sugaku = c(75, 88, 95)
)
write.csv(df, "scores_r.csv", row.names = FALSE)   # 行番号は保存しない
cat("scores_r.csv に書き込みました\n")

In [ ]:
df2 <- read.csv("scores_r.csv")
print(df2)
str(df2)                              # 数値の列は自動的に数値として読み込まれる
df2$total <- df2$kokugo + df2$sugaku
df2

### 11.3 ファイルの存在確認と一覧

In [ ]:
print(file.exists("sample_r.txt"))
print(file.exists("not_found.txt"))
print(list.files())                          # 現在のフォルダにあるファイル
getwd()                               # 現在のフォルダ

### 練習問題 11

1. `"memo_r.txt"` に 3 行のメモを書き込み、読み込んで行数を表示してください。
2. `"scores_r.csv"` を読み込み、各人の合計点と、全員の合計点の平均を表示してください。

In [ ]:
# 練習問題 11 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 11 の解答例を見る</strong></summary>

```r
# 1
writeLines(c("牛乳を買う", "レポートを提出する", "R を復習する"), "memo_r.txt")
memo <- readLines("memo_r.txt")
print(length(memo))

# 2
df <- read.csv("scores_r.csv")
df$total <- df$kokugo + df$sugaku
print(df[, c("name", "total")])
mean(df$total)
```

</details>

---
## 12. グラフの基本

R には最初からグラフ描画の機能（`graphics` パッケージ）が備わっています。
0.3 節で説明したとおり、JupyterLite の R 環境では **グラフの中の日本語は表示できない** ので、
タイトルや軸ラベルは英語で書きます。まず、グラフの大きさを設定しておきます。

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 100, jupyter.plot_scale = 1)

### 12.1 折れ線グラフ・散布図：plot()

`plot(x, y)` が基本です。`type = "l"` で線、`"p"` で点（既定）、`"b"` で両方になります。

In [ ]:
x <- 1:10
y <- x ^ 2
plot(x, y, type = "b", main = "y = x^2", xlab = "x", ylab = "y", col = "steelblue", pch = 16)

In [ ]:
# 散布図と回帰直線
set.seed(1)
study_hours <- runif(30, 0, 10)
scores <- 50 + 4 * study_hours + rnorm(30, sd = 5)
plot(study_hours, scores, main = "Study hours vs. score", xlab = "Study hours", ylab = "Score", pch = 16)
abline(lm(scores ~ study_hours), col = "red", lwd = 2)    # 回帰直線を重ねる

### 12.2 棒グラフ：barplot()

In [ ]:
sales <- c(Tokyo = 120, Nagoya = 95, Osaka = 110, Fukuoka = 70)
barplot(sales, main = "Sales by city", ylab = "Sales (10k yen)", col = "steelblue")

### 12.3 ヒストグラムと箱ひげ図：hist()、boxplot()

In [ ]:
set.seed(42)
heights <- rnorm(200, mean = 170, sd = 6)
hist(heights, main = "Histogram of heights", xlab = "Height (cm)", col = "lightgray", breaks = 15)

In [ ]:
boxplot(Sepal.Length ~ Species, data = iris, main = "Sepal length by species", xlab = "Species", ylab = "Sepal length (cm)")

### 12.4 複数のグラフを並べる：par(mfrow)

In [ ]:
par(mfrow = c(1, 2))              # 1 行 2 列に並べる
plot(x, y, type = "l", main = "Line")
barplot(sales, main = "Bar")
par(mfrow = c(1, 1))              # 元に戻す

### 練習問題 12

1. `x <- 1:12` と `y <- c(5, 7, 9, 12, 15, 18, 22, 21, 17, 13, 9, 6)` を折れ線グラフ（`type = "l"`）にしてください。タイトルは "Monthly temperature" にします。
2. `iris` の `Petal.Length` のヒストグラムを描いてください。

In [ ]:
# 練習問題 12 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 12 の解答例を見る</strong></summary>

```r
# 1
x <- 1:12
y <- c(5, 7, 9, 12, 15, 18, 22, 21, 17, 13, 9, 6)
plot(x, y, type = "l", main = "Monthly temperature", xlab = "Month", ylab = "Temperature (C)")

# 2
hist(iris$Petal.Length, main = "Histogram of petal length", xlab = "Petal length (cm)")
```

</details>

---
## 13. まとめ

このチュートリアルで学んだことをまとめます。

| トピック | 主な文法・関数 |
|---|---|
| JupyterLite の準備 | `library()`, `installed.packages()`, `requireNamespace()`, `options(repr.plot.*, jupyter.plot_scale = 1)`, `Sys.setenv(TZ = ...)` |
| 基本 | `print()`, `cat()`, `#` コメント, 算術演算子 `+ - * / %/% %% ^` |
| 変数とデータ型 | `<-`, `numeric`, `character`, `logical`, `class()`, `as.numeric()`, `NA`, `is.na()` |
| ベクトル | `c()`, `1:10`, `seq()`, `rep()`, 1 始まりのインデックス, 論理インデックス, `sum()`, `mean()`, `sort()` |
| 文字列 | `paste()`, `nchar()`, `substr()`, `strsplit()`, `gsub()`, `grepl()`, `sprintf()` |
| リスト・データフレーム | `list()`, `$`, `data.frame()`, `head()`, `str()`, `df[行, 列]`, `order()`, `factor()` |
| 条件分岐 | `if / else if / else`, `&&`, `||`, `ifelse()`, `switch()` |
| 繰り返し | `for`, `while`, `break`, `next`, `seq_along()`, `sapply()`, `apply()` |
| 関数 | `function()`, `return()`, デフォルト引数, 名前付き引数, 無名関数 |
| エラー処理 | `tryCatch()`, `stop()`, `warning()`, `suppressWarnings()` |
| 統計・日付 | `set.seed()`, `rnorm()`, `sample()`, `mean()`, `sd()`, `summary()`, `cor()`, `Sys.Date()`, `as.Date()` |
| ファイル | `writeLines()`, `readLines()`, `write.csv()`, `read.csv()`, `list.files()` |
| グラフ | `plot()`, `barplot()`, `hist()`, `boxplot()`, `abline()`, `par(mfrow)` |

## 次のステップ

R の基本が身についたら、統計解析の演習に進みましょう。

1. `jupyterlite/jupyterlite_xeus_r_stats_practice.ipynb` — t 検定、カイ二乗検定、分散分析、回帰分析（R）
2. `python/python_beginner_tutorial.ipynb` — Python でも同じことをやってみる（日本語グラフはこちらで）

---
## 総合演習：成績管理プログラム

これまで学んだ内容を組み合わせて、次の課題に挑戦してください。

次のデータフレームは、4 人の学生の 3 科目の点数です。

```r
scores <- data.frame(
  name = c("田中", "鈴木", "佐藤", "高橋"),
  kokugo = c(78, 88, 95, 55),
  sugaku = c(92, 64, 89, 71),
  eigo = c(85, 71, 93, 60)
)
```

1. 平均点から評価を返す関数 `grade(avg)` を作ってください（85 以上「優」、70 以上「良」、60 以上「可」、それ未満「不可」）。
2. 学生ごとに合計点・平均点・評価を計算して列として追加し、`田中: 合計 255 点, 平均 85.0 点, 評価 優` の形式で表示してください。
3. 科目ごとの平均点を計算し、最も平均点が高い科目を表示してください。
4. 結果のデータフレームを `results_r.csv` に保存してください。
5. 学生ごとの平均点を棒グラフにしてください（ラベルは英語またはローマ字で）。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください
scores <- data.frame(
  name = c("田中", "鈴木", "佐藤", "高橋"),
  kokugo = c(78, 88, 95, 55),
  sugaku = c(92, 64, 89, 71),
  eigo = c(85, 71, 93, 60)
)

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 100, jupyter.plot_scale = 1)

scores <- data.frame(
  name = c("田中", "鈴木", "佐藤", "高橋"),
  kokugo = c(78, 88, 95, 55),
  sugaku = c(92, 64, 89, 71),
  eigo = c(85, 71, 93, 60)
)

# 1. 評価を返す関数
grade <- function(avg) {
  if (avg >= 85) {
    "優"
  } else if (avg >= 70) {
    "良"
  } else if (avg >= 60) {
    "可"
  } else {
    "不可"
  }
}

# 2. 学生ごとの集計
subject_cols <- c("kokugo", "sugaku", "eigo")
scores$total <- rowSums(scores[, subject_cols])
scores$avg <- round(scores$total / length(subject_cols), 1)
scores$grade <- sapply(scores$avg, grade)
for (i in seq_len(nrow(scores))) {
  cat(sprintf("%s: 合計 %d 点, 平均 %.1f 点, 評価 %s\n",
              scores$name[i], scores$total[i], scores$avg[i], scores$grade[i]))
}

# 3. 科目ごとの平均
subject_avg <- colMeans(scores[, subject_cols])
print(subject_avg)
cat("最も平均点が高い科目:", names(which.max(subject_avg)), "\n")

# 4. CSV に保存
write.csv(scores, "results_r.csv", row.names = FALSE)
cat("results_r.csv に保存しました\n")

# 5. 棒グラフ（ラベルはローマ字で）
labels <- c("Tanaka", "Suzuki", "Sato", "Takahashi")
barplot(scores$avg, names.arg = labels, main = "Average score by student",
        ylab = "Average score", ylim = c(0, 100), col = "steelblue")
abline(h = mean(scores$avg), col = "red", lty = 2, lwd = 2)
legend("topright", legend = sprintf("Overall mean %.1f", mean(scores$avg)), col = "red", lty = 2, lwd = 2)

お疲れさまでした！ ここまでできれば、R の基本文法はひととおり身についています。
「次のステップ」の統計テスト演習に進んで、t 検定や回帰分析に挑戦してみましょう。